# Chapter 11 companion: stability of sampled-data systems.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.

Reproduces:
  * the sampled integrator under feedback, H(z) = T/(z - (1 - KT)), T = 0.1 s:
    sum |h| for K = 5, 10, 15, 25; stable for 0 < K < 2/T;
  * Routh's array (with the epsilon rule and the auxiliary polynomial), on the
    chapter's examples: s^4 + s^3 + 2s^2 + 3s + 2, the gain range 0 < K < 204/25,
    s^3 - 7s + 6, s^3 + 3s^2 + 4s + 12, and s^5 + s^4 + 3s^3 + 3s^2 - 4s - 4;
  * the bilinear substitution Q_w(w) = (1-w)^n Q((1+w)/(1-w)) followed by Routh,
    and Jury's table, both on
      ZOH + 1/(s(s+1)), T = 1 s, gain K:            0 < K < 2.39
      the same loop with one sample of delay:       0 < K < 0.805
      the laser loop with D = k:                    -2 < k < 54.7 (T = 0.1 s)
      the exit ticket, accumulator on 0.5/(z-0.5):  0 < K < 6
    every range is cross-checked with the roots of the characteristic polynomial.

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from numpy.polynomial import polynomial as P
from scipy import signal
from scipy.optimize import brentq

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

In [ ]:
# ---------------------------------------------------------------- tools
def routh(c, eps=1e-6, verbose=False, var="s"):
    """Routh array of the polynomial with descending coefficients c.
    Returns the first column and the number of sign changes (roots in Re s > 0)."""
    c = np.array(c, float)
    n = len(c) - 1
    width = n // 2 + 1
    pad = lambda r: np.r_[r, np.zeros(width - len(r))]
    rows = [pad(c[0::2]), pad(c[1::2])]
    notes = []
    for r in range(1, n + 1):
        cur = rows[r]
        if np.all(np.abs(cur) < 1e-12):                     # a row of zeros
            up, d = rows[r - 1], n - r + 1
            aux = [float(up[j]) for j in range(width) if d - 2 * j >= 0]
            notes.append(f"row {var}^{n - r} is zero: auxiliary P(s) of degree {d}, coefficients {aux}")
            rows[r] = cur = pad(np.array([up[j] * (d - 2 * j) for j in range(width) if d - 2 * j > 0]))
        elif abs(cur[0]) < 1e-12:                           # a zero first entry
            notes.append(f"zero first entry in row {var}^{n - r}: replaced by eps = {eps}")
            cur = cur.copy()
            cur[0] = eps
            rows[r] = cur
        if r == n:
            break
        up = rows[r - 1]
        rows.append(np.array([(cur[0] * up[j + 1] - up[0] * cur[j + 1]) / cur[0]
                              for j in range(width - 1)] + [0.0]))
    first = np.array([row[0] for row in rows[:n + 1]])
    changes = int(np.sum(np.sign(first[:-1]) != np.sign(first[1:])))
    if verbose:
        for r, row in enumerate(rows[:n + 1]):
            print(f"    {var}^{n - r}: " + "  ".join(f"{v:9.4g}" for v in row))
        for t in notes:
            print("    " + t)
    return first, changes

In [ ]:
def bilinear(c):
    """Q_w(w) = (1-w)^n Q((1+w)/(1-w)) for Q with descending coefficients c."""
    n = len(c) - 1
    out = np.zeros(n + 1)
    for i, ci in enumerate(c):
        k = n - i
        out += ci * P.polymul(P.polypow([1, 1], k), P.polypow([1, -1], n - k))[:n + 1]
    return out[::-1]

In [ ]:
def jury(c):
    """Jury's test on descending coefficients (leading coefficient > 0)."""
    a = np.array(c[::-1], float)                 # a0, a1, ..., an
    n = len(a) - 1
    if not (np.sum(a) > 0 and (-1) ** n * np.polyval(c, -1) > 0 and abs(a[0]) < a[-1]):
        return False
    row = a
    while len(row) > 3:
        m = len(row) - 1
        row = np.array([row[0] * row[j] - row[m] * row[m - j] for j in range(m)])
        if not abs(row[0]) > abs(row[-1]):
            return False
    return True

In [ ]:
def stable_by_roots(c):
    return bool(np.all(np.abs(np.roots(c)) < 1))

In [ ]:
def gain_limit(qfun, lo, hi, test):
    """Largest gain in (lo, hi) at which test(qfun(K)) still holds (bisection on the edge)."""
    return brentq(lambda K: (1.0 if test(qfun(K)) else -1.0), lo, hi, xtol=1e-10)

In [ ]:
# ---------------------------------------------------------------- the sampled integrator
T = 0.1

In [ ]:
print("Sampled integrator T/(z-1) with feedback K, T = 0.1 s: H(z) = T/(z - (1 - KT))")

In [ ]:
for K in (5, 10, 15, 25):
    pole = 1 - K * T
    h = T * pole ** np.arange(200.0)
    s = np.sum(np.abs(h))
    print(f"  K = {K:2d}: pole {pole:5.2f},  sum |h| = {s:.3g}" + ("  (grows)" if abs(pole) >= 1 else ""))

In [ ]:
print(f"  stable for 0 < K < 2/T = {2 / T:g}")

In [ ]:
# ---------------------------------------------------------------- Routh examples
print("\nRouth: s^4 + s^3 + 2s^2 + 3s + 2")

In [ ]:
_, ch = routh([1, 1, 2, 3, 2], verbose=True)
print(f"  sign changes: {ch}; roots {np.round(np.roots([1, 1, 2, 3, 2]), 3)}")
Kmax = gain_limit(lambda K: [1, 5, 8, 6, K], 0.1, 12, lambda c: routh(c)[1] == 0)
print(f"Gain range of s^4 + 5s^3 + 8s^2 + 6s + K: 0 < K < {Kmax:.4f} (204/25 = {204 / 25:.4f})")
print(f"  at K = 204/25 the roots on the axis: {np.round(np.roots([1, 5, 8, 6, 204 / 25]), 3)}")

In [ ]:
for c in ([1, 0, -7, 6], [1, 3, 4, 12], [1, 1, 3, 3, -4, -4]):
    print("Routh:", c)
    _, ch = routh(c, verbose=True)
    print(f"  right half-plane roots: {ch};  roots {np.round(np.roots(c), 3)}")

In [ ]:
# ---------------------------------------------------------------- sampled loops
e1 = np.exp(-1.0)

In [ ]:
# laser loop: exact ZOH equivalent of 2/((s+1)(s+4)) with T = 0.1 s
nl, dl, _ = signal.cont2discrete(([2], [1, 5, 4]), 0.1, method="zoh")

In [ ]:
nl = np.atleast_2d(nl)[0][-2:]
print(f"\nLaser loop, T = 0.1 s: G(z) = ({nl[0]:.4f} z + {nl[1]:.4f})/(z^2 - {-dl[1]:.4f} z + {dl[2]:.4f})")
loops = {
    "ZOH + 1/(s(s+1)), T = 1 s, gain K":
        (lambda K: [1, e1 * K - (1 + e1), e1 + (1 - 2 * e1) * K], 0.01, 10),
    "same loop, one sample of delay (D = K/z)":
        (lambda K: [1, -(1 + e1), e1 * (1 + K), (1 - 2 * e1) * K], 0.01, 3),
    "laser loop, T = 0.1 s, D = k":
        (lambda K: [1, dl[1] + nl[0] * K, dl[2] + nl[1] * K], 0.01, 200),
    "exit ticket: D = Kz/(z-1), G = 0.5/(z-0.5)":
        (lambda K: [1, 0.5 * K - 1.5, 0.5], 0.01, 20),
}
print("\nUpper gain limits, three ways")

In [ ]:
for name, (q, lo, hi) in loops.items():
    k_roots = gain_limit(q, lo, hi, stable_by_roots)
    k_bil = gain_limit(q, lo, hi, lambda c: routh(bilinear(c))[1] == 0 and np.all(bilinear(c) > 0))
    k_jury = gain_limit(q, lo, hi, jury)
    print(f"  {name:45s}: roots {k_roots:.4f}, bilinear + Routh {k_bil:.4f}, Jury {k_jury:.4f}")

In [ ]:
print("\nThe delayed loop, K = 0.5: Q_w(w) =", np.round(bilinear(loops["same loop, one sample of delay (D = K/z)"][0](0.5)), 4))
routh(bilinear(loops["same loop, one sample of delay (D = K/z)"][0](0.5)), verbose=True, var="w")

In [ ]:
# laser loop: the exact ZOH coefficients used above come from G(z) of 2/((s+1)(s+4)), T = 0.1 s
print("Laser loop lower limit: Q(1) = 0 at k =", round(brentq(lambda k: np.polyval(
    loops["laser loop, T = 0.1 s, D = k"][0](k), 1), -10, 0), 3))

In [ ]:
# ---------------------------------------------------------------- figure
Ks = np.linspace(0.01, 3.0, 600)

In [ ]:
r0 = [np.max(np.abs(np.roots(loops["ZOH + 1/(s(s+1)), T = 1 s, gain K"][0](K)))) for K in Ks]
r1 = [np.max(np.abs(np.roots(loops["same loop, one sample of delay (D = K/z)"][0](K)))) for K in Ks]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(Ks, r0, label="D = K")
axes[0].plot(Ks, r1, label="D = K/z (one sample of delay)")
axes[0].axhline(1, color="k", lw=0.6)
axes[0].set(xlabel="K", ylabel="largest |pole|", title="ZOH + 1/(s(s+1)), T = 1 s")
axes[0].legend(fontsize=8)
tri = np.array([[-2, 1], [2, 1], [0, -1], [-2, 1]])
axes[1].fill(tri[:, 0], tri[:, 1], alpha=0.2, label="stable: |a0| < 1, Q(1) > 0, Q(-1) > 0")
Kp = np.linspace(0, 3, 50)
axes[1].plot(e1 * Kp - (1 + e1), e1 + (1 - 2 * e1) * Kp, "r", label="(a1, a0) as K goes 0 to 3")
axes[1].set(xlabel="a1", ylabel="a0", title="Second-order test for z^2 + a1 z + a0")
axes[1].legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "ch11_stability.png", dpi=150)
print("\nSaved figures/ch11_stability.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**Where is the running loop stable?** A zero-order hold drives $G_p(s) = 1/(s(s+1))$, with
$D(z) = K$ (or $K/z$, one sample of computation delay) and unity feedback. The left panel is the
stable region in the $(T, K)$ plane, from Jury's test; the orange dot is your choice. At $T = 1$ s
the loop is stable for $0 < K < 2.39$, and only for $0 < K < 0.805$ with the delay. On the right,
the second-order test: the point $(a_1, a_0)$ of $z^2 + a_1 z + a_0$ must stay inside the triangle.
Shrink $T$ and watch the gain limit grow, roughly like $2/T$.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
NO_DELAY, DELAY = "D(z) = K", "D(z) = K/z"


def _coefficients(K, T, controller):
    """Characteristic polynomial of 1 + D G = 0, descending powers of z."""
    p = np.exp(-T)
    b1, b0 = T - 1 + p, 1 - p - T * p                  # G(z) = (b1 z + b0)/((z - 1)(z - p))
    if controller == NO_DELAY:
        return np.array([1.0, K * b1 - (1 + p), p + K * b0])
    return np.array([1.0, -(1 + p), p + K * b1, K * b0])


def _jury(c):
    """Jury's conditions for a monic polynomial of degree 2 or 3: (values, all hold?)."""
    q1, qm1 = np.polyval(c, 1.0), (-1) ** (len(c) - 1) * np.polyval(c, -1.0)
    a0 = c[-1]
    cond = [q1 > 0, qm1 > 0, abs(a0) < 1]
    if len(c) == 4:                                    # z^3 + a2 z^2 + a1 z + a0
        cond.append(abs(a0 ** 2 - 1) > abs(a0 * c[1] - c[2]))
    return (q1, qm1, a0), all(cond)


def _k_limit(T, controller):
    """Largest K with every condition of Jury's test met (bisection; stable for 0 < K < limit)."""
    lo, hi = 1e-6, 4.0 / T + 5.0
    for _ in range(32):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if _jury(_coefficients(mid, T, controller))[1] else (lo, mid)
    return lo


def explore(K=1.0, T=1.0, controller=NO_DELAY):
    c = _coefficients(K, T, controller)
    (q1, qm1, a0), stable = _jury(c)
    roots = np.roots(c)
    k_lim = _k_limit(T, controller)

    Ts = np.linspace(0.05, 3.0, 60)
    Ks = np.array([_k_limit(t, controller) for t in Ts])
    k_top = max(6.0, 1.2 * K)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.5, 1]})
    ax1.fill_between(Ts, 0, np.minimum(Ks, k_top), color=TEAL, alpha=0.18, label="stable")
    ax1.plot(Ts, Ks, color=TEAL, lw=2, label="gain limit")
    if controller == DELAY:
        ax1.plot(Ts, [_k_limit(t, NO_DELAY) for t in Ts], "--", color=GRAY, lw=1.4,
                 label="limit without the delay")
    ax1.plot([T], [K], "o" if stable else "X", color=ORANGE, ms=11, mec="k",
             label=f"K = {K:g}, T = {T:g} s")
    ax1.set_xlim(0, 3.0)
    ax1.set_ylim(0, k_top)
    ax1.set_xlabel("T (s)")
    ax1.set_ylabel("K")
    ax1.set_title(f"{controller}: stable region from Jury's test", fontsize=11)
    ax1.legend(fontsize=10, loc="upper right")

    if controller == NO_DELAY:
        tri = np.array([[-2, 1], [2, 1], [0, -1], [-2, 1]])
        ax2.fill(tri[:, 0], tri[:, 1], color=TEAL, alpha=0.18)
        ax2.plot(tri[:, 0], tri[:, 1], color=TEAL, lw=1.5)
        kk = np.linspace(0, max(k_lim, K) * 1.3, 60)
        path = np.array([_coefficients(k, T, controller) for k in kk])
        ax2.plot(path[:, 1], path[:, 2], color=GRAY, lw=1.2, label="K from 0 up")
        ax2.plot([c[1]], [c[2]], "o" if stable else "X", color=ORANGE, ms=11, mec="k",
                 label="($a_1$, $a_0$) now")
        ax2.set_xlim(-2.6, 2.6)
        ax2.set_ylim(-1.4, 1.9)
        ax2.set_xlabel("$a_1$")
        ax2.set_ylabel("$a_0$")
        ax2.set_title("$|a_0| < 1$, $Q(1) > 0$, $Q(-1) > 0$", fontsize=11)
        ax2.legend(fontsize=10, loc="upper left")
    else:
        th = np.linspace(0, 2 * np.pi, 300)
        ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
        ax2.plot(roots.real, roots.imag, "x", color=ORANGE, ms=11, mew=2.5)
        ax2.axhline(0, color="k", lw=0.5)
        ax2.axvline(0, color="k", lw=0.5)
        lim = max(1.3, 1.1 * np.max(np.abs(roots)))
        ax2.set_xlim(-lim, lim)
        ax2.set_ylim(-lim, lim)
        ax2.set_aspect("equal")
        ax2.set_xlabel("Re z")
        ax2.set_ylabel("Im z")
        ax2.set_title("closed-loop poles (cubic)", fontsize=11)
    fig.tight_layout()

    powers = ["z^3", "z^2", "z", ""][4 - len(c):]
    print("Q(z) = " + " ".join(f"{v:+.4f} {w}" for v, w in zip(c, powers)).strip())
    print(f"Jury: Q(1) = {q1:.4f}, (-1)^n Q(-1) = {qm1:.4f}, |a0| = {abs(a0):.4f}"
          + ("" if len(c) == 3 else f", |a0^2 - 1| = {abs(a0 ** 2 - 1):.4f} vs |a0 a2 - a1| = {abs(a0 * c[1] - c[2]):.4f}"))
    if len(c) == 3:
        print(f"bilinear: Q_w(w) = {qm1:.4f} w^2 {2 * (1 - a0):+.4f} w {q1:+.4f}  (all > 0 means stable)")
    print(("stable" if stable else "NOT stable") + f";  largest |z| = {np.max(np.abs(roots)):.3f};"
          f"  at T = {T:g} s the loop is stable for 0 < K < {k_lim:.3f}")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(explore,
             K=FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05, description="K",
                           continuous_update=False),
             T=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.05, description="T (s)",
                           continuous_update=False),
             controller=Dropdown(options=[NO_DELAY, DELAY], value=NO_DELAY, description="D(z)"))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 11.** These cells go with the Hands-on exercises at the end of
Chapter 11. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
def unit_circle(ax, r=None):
    c = np.linspace(0, 2 * np.pi, 300)
    ax.plot(np.cos(c), np.sin(c), color=GRAY, lw=1)
    if r is not None:
        ax.plot(r * np.cos(c), r * np.sin(c), ":", color=TEAL, lw=1.2)
    ax.set_aspect("equal")
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.25, 1.25)

In [ ]:
# ---------------------------------------------------------------- H1
def h1_worst_input(K=3.0):
    """H1: y(k+1) = 0.6 y(k) + 0.4 u(k), u = K (r - y); a step and the worst-case reference."""
    n = 30
    pole = 0.6 - 0.4 * K
    h = np.r_[0.0, 0.4 * K * pole ** np.arange(n - 1)]       # impulse response
    bound = np.sum(np.abs(h)) if abs(pole) < 1 else np.inf

    def run(r):
        y = np.zeros(n + 1)
        for k in range(n):
            y[k + 1] = 0.6 * y[k] + 0.4 * K * (r[k] - y[k])
        return y

    step = np.ones(n)
    alt = np.where(np.arange(n) % 2 == 0, 1.0, -1.0)        # alternating reference
    ys, ya = run(step), run(alt)
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(np.arange(n + 1), ys, "o-", color=BLUE, ms=4, lw=1, label=f"step: peak {np.max(np.abs(ys)):.2f} A")
    ax.plot(np.arange(n + 1), ya, "s-", color=ORANGE, ms=4, lw=1,
            label=f"alternating reference: peak {np.max(np.abs(ya)):.2f} A")
    if np.isfinite(bound):
        ax.axhline(bound, color=TEAL, ls="--", lw=1, label=f"sum |h| = {bound:.2f} A")
        ax.axhline(-bound, color=TEAL, ls="--", lw=1)
    ax.set_ylim(-4, 4)
    ax.set_xlabel("k")
    ax.set_ylabel("current y (A)")
    ax.set_title(f"K = {K:.2f}: closed-loop pole {pole:.2f}", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_heater(K=0.2):
    """H2: Q(z) = z^3 - 2.4 z^2 + (1.85 + 0.1K) z - 0.45; poles and the step response."""
    Q = [1, -2.4, 1.85 + 0.1 * K, -0.45]
    p = np.roots(Q)
    n = 150
    # closed loop Y/R = 0.1 K z / Q(z): y(k) = 2.4 y(k-1) - (1.85+0.1K) y(k-2) + 0.45 y(k-3) + 0.1 K r(k-2)
    y = np.zeros(n)
    for k in range(2, n):
        y[k] = 2.4 * y[k - 1] - (1.85 + 0.1 * K) * y[k - 2] \
            + 0.45 * (y[k - 3] if k >= 3 else 0.0) + 0.1 * K
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(np.arange(n), y, color=ORANGE, lw=1.4)
    ax1.axhline(1, color=GRAY, lw=0.6)
    ax1.set_ylim(-1, 3)
    ax1.set_xlabel("t (s)")
    ax1.set_title(f"K = {K:.3f}: largest |pole| {max(abs(p)):.4f}", fontsize=11)
    unit_circle(ax2)
    ax2.plot(p.real, p.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax2.set_title("closed-loop poles", fontsize=10)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_pi_triangle(Kp=5.0, Ki=100.0):
    """H3: drone altitude, G(z) = T/(z-1), PI D(z) = Kp + Ki T z/(z-1), T = 0.1 s."""
    T = 0.1
    x, y = T * Kp, T ** 2 * Ki
    p = np.roots([1, x + y - 2, 1 - x])
    n = 60
    h = np.zeros(n + 1)
    acc, href = 0.0, 1.0
    for k in range(n):
        e = href - h[k]
        acc += e
        u = Kp * e + Ki * T * acc
        h[k + 1] = h[k] + T * u
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 3.8), gridspec_kw={"width_ratios": [2, 1.2, 1]})
    ax1.plot(T * np.arange(n + 1), h, "o-", color=ORANGE, ms=3, lw=1)
    ax1.axhline(1, color=GRAY, lw=0.6)
    ax1.set_ylim(-1, 3)
    ax1.set_xlabel("t (s)")
    ax1.set_ylabel("altitude (m)")
    ax1.set_title(f"Kp = {Kp:g}, Ki = {Ki:g}: largest |pole| {max(abs(p)):.3f}", fontsize=11)
    ax2.fill([0, 20, 0], [0, 0, 400], color=BLUE, alpha=0.15)
    ax2.plot([0, 20, 0, 0], [0, 0, 400, 0], color=BLUE, lw=1)
    ax2.plot(Kp, Ki, "o", color=ORANGE, ms=8)
    ax2.set_xlim(-3, 25)
    ax2.set_ylim(-30, 450)
    ax2.set_xlabel("Kp (1/s)")
    ax2.set_ylabel("Ki (1/s$^2$)")
    ax2.set_title("stable region", fontsize=10)
    unit_circle(ax3)
    ax3.plot(p.real, p.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax3.set_title("poles", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_gimbal(K=1.1):
    """H4: G(z) = 0.1(z+1)/((z-1)(z-0.6)), T = 0.05 s, D = K; target pans at 0.5 rad/s."""
    T, rate = 0.05, 0.5
    p = np.roots([1, 0.1 * K - 1.6, 0.6 + 0.1 * K])
    p2 = np.roots([1, 0.2 * K - 1.6, 0.6 + 0.2 * K])          # the loop at 2K
    n = 120
    y = np.zeros(n + 1)
    e = np.zeros(n + 1)
    for k in range(n + 1):
        e[k] = rate * T * k - y[k]
        if k + 1 <= n:
            # y(k+1) = 1.6 y(k) - 0.6 y(k-1) + 0.1 K (e(k) + e(k-1))
            y[k + 1] = 1.6 * y[k] - 0.6 * (y[k - 1] if k >= 1 else 0.0) \
                + 0.1 * K * (e[k] + (e[k - 1] if k >= 1 else 0.0))
    ok = K >= 1 and max(abs(p2)) < 1 and max(abs(p)) <= 0.85 + 1e-9
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(T * np.arange(n + 1), e, color=ORANGE, lw=1.4, label="tracking error (rad)")
    ax1.axhline(0.05, color=TEAL, ls="--", lw=1, label="spec 0.05 rad")
    ax1.set_ylim(-0.05, 0.2)
    ax1.set_xlabel("t (s)")
    ax1.set_title(f"K = {K:.3f}: |z| max {max(abs(p)):.3f}, at 2K {max(abs(p2)):.3f}: "
                  f"{'PASS' if ok else 'FAIL'}", fontsize=11)
    ax1.legend(loc="upper right", fontsize=9)
    unit_circle(ax2, r=0.85)
    ax2.plot(p.real, p.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax2.plot(p2.real, p2.imag, "+", color=BLUE, ms=10, mew=1.5)
    ax2.set_title("poles at K (x) and 2K (+)", fontsize=10)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    interact(h1_worst_input, K=FloatSlider(value=3.0, min=0.1, max=3.9, step=0.05, description="K",
                                           continuous_update=False))
    interact(h2_heater, K=FloatSlider(value=0.2, min=0.01, max=2.0, step=0.005, readout_format=".3f",
                                      description="K", continuous_update=False))
    interact(h3_pi_triangle,
             Kp=FloatSlider(value=5.0, min=-2.0, max=24.0, step=0.5, description="Kp (1/s)",
                            continuous_update=False),
             Ki=FloatSlider(value=100.0, min=-20.0, max=440.0, step=5.0, description="Ki (1/s^2)",
                            continuous_update=False))
    interact(h4_gimbal, K=FloatSlider(value=1.1, min=0.05, max=4.2, step=0.01, readout_format=".2f",
                                      description="K", continuous_update=False))
except ImportError:
    h1_worst_input()
    h2_heater()
    h3_pi_triangle()
    h4_gimbal()